# Práctica 4 — Evaluación y Validación de Modelos
## Dataset: Palmer Penguins

**ISTER · Sistemas y Gestión de Data · Quinto Nivel · 2026**  
Ing. David Minango. PhD

---

En esta práctica evaluarás modelos de clasificación usando el dataset **Palmer Penguins** — 344 pingüinos de 3 especies medidos en la Antártida. Aplicarás todas las métricas vistas en clase: matriz de confusión, F1-Score, Cross-Validation y Curva ROC.

**Regla:** Las celdas marcadas con `# 🔧 TU CÓDIGO` debes completarlas. Las demás solo ejecútalas sin modificar.

## Parte 0 — Setup

Ejecuta esta celda sin modificar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.metrics         import (confusion_matrix, classification_report,
                                     ConfusionMatrixDisplay, roc_curve, auc)
from sklearn.preprocessing   import label_binarize

print('✅ Librerías cargadas')

---
## Parte 1 — Carga y Preparación del Dataset

El dataset Penguins viene incluido en seaborn. Tiene valores nulos que debes eliminar antes de modelar.

In [ ]:
# DADO: cargar y explorar
df = sns.load_dataset('penguins')
print(f'Forma original: {df.shape}')
print(f'\nColumnas: {df.columns.tolist()}')
print(f'\nValores nulos por columna:')
print(df.isnull().sum())
print(f'\nDistribución de especies:')
print(df['species'].value_counts())

In [ ]:
# 🔧 TU CÓDIGO
# 1. Elimina filas con valores nulos usando dropna()
# 2. Selecciona las 4 columnas numéricas como X:
#    'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g'
# 3. Convierte 'species' a código numérico (0, 1, 2) como y
#    Usa: pd.Categorical(df_limpio['species']).codes
# 4. Imprime el nuevo shape y clases únicas

df_limpio = ___________
X = ___________
y = ___________

nombres_especies = df_limpio['species'].cat.categories.tolist()
print(f'Forma limpia: {df_limpio.shape}')
print(f'Clases (código → especie): {list(enumerate(nombres_especies))}')
print(f'Distribución: {np.unique(y, return_counts=True)}')

In [ ]:
# 🔧 TU CÓDIGO
# Crea un pairplot con seaborn del dataframe limpio
# Usa hue='species' para colorear por especie
# ¿Qué variable separa mejor a las 3 especies visualmente?

# ___ tu código aquí ___

**❓ Responde aquí (celda Markdown):**

1. ¿Qué par de variables separa mejor a Gentoo del resto? ¿Por qué tiene sentido biológicamente?
2. ¿Adelie y Chinstrap son fáciles de separar? ¿Qué variable los diferencia mejor?

---
## Parte 2 — Entrenamiento y Métricas Básicas

In [ ]:
# 🔧 TU CÓDIGO
# Divide X e y: 70% train, 30% test, random_state=42, stratify=y
# Entrena:
#   - DecisionTreeClassifier(max_depth=5, random_state=42)
#   - RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
# Imprime train accuracy y test accuracy de cada modelo

X_train, X_test, y_train, y_test = ___________
arbol = ___________
rf    = ___________

# ___ entrenar y mostrar resultados ___

In [ ]:
# 🔧 TU CÓDIGO
# Crea una figura con 2 subplots lado a lado
# En cada uno: ConfusionMatrixDisplay.from_predictions(...)
# Usa nombres_especies como display_labels
# Título: 'Árbol de Decisión' y 'Random Forest'

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# ___ tu código aquí ___
plt.tight_layout()
plt.show()

In [ ]:
# 🔧 TU CÓDIGO
# Imprime el classification_report del Random Forest
# Usa target_names=nombres_especies
# ¿Qué especie tiene el F1-Score más bajo?

y_pred_rf = rf.predict(X_test)
print('── Classification Report — Random Forest ────────────')
# ___ tu código aquí ___

**❓ Responde aquí:**

1. ¿Cuánto mejoró el F1 al pasar de árbol a Random Forest?
2. ¿Qué especie tiene más errores en la matriz? ¿Con cuál se confunde?
3. El Support de Chinstrap es menor — ¿cómo afecta eso a la confianza en su F1?

---
## Parte 3 — Cross-Validation 10-Fold

In [ ]:
# 🔧 TU CÓDIGO
# Calcula cross_val_score con cv=10 y scoring='f1_weighted' para árbol y RF
# Imprime media y desviación estándar de cada uno

cv_arbol = ___________
cv_rf    = ___________

print(f'Árbol  | F1 media: {cv_arbol.mean():.4f} | Std: {cv_arbol.std():.4f}')
print(f'RF 100 | F1 media: {cv_rf.mean():.4f} | Std: {cv_rf.std():.4f}')

In [ ]:
# 🔧 TU CÓDIGO
# Crea un boxplot que compare los 10 scores del árbol vs RF
# plt.boxplot([cv_arbol, cv_rf], labels=['Árbol', 'RF 100'])
# Incluye: título, ylabel='F1-Score', grid

# ___ tu código aquí ___

**❓ Responde aquí:**

1. ¿Cuál modelo tiene mayor F1 promedio? ¿Y cuál tiene menor variabilidad?
2. ¿Hay algún fold donde el árbol supera al RF? ¿Qué puede explicarlo?
3. ¿Por qué usamos F1-weighted en lugar de accuracy para este dataset?

---
## Parte 4 — Curvas ROC One-vs-Rest (Multiclase)

In [ ]:
# DADO: binarizar etiquetas para ROC multiclase
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
y_prob_rf  = rf.predict_proba(X_test)
print(f'y_test_bin shape: {y_test_bin.shape}')
print(f'y_prob_rf  shape: {y_prob_rf.shape}')

In [ ]:
# 🔧 TU CÓDIGO
# Para cada clase i en [0, 1, 2]:
#   - Calcula fpr, tpr con roc_curve(y_test_bin[:, i], y_prob_rf[:, i])
#   - Calcula el AUC con auc(fpr, tpr)
#   - Dibuja la curva con etiqueta f'{nombres_especies[i]} (AUC = {roc_auc:.3f})'
# Añade la línea diagonal punteada de referencia

colores = ['#3b82f6', '#10b981', '#f59e0b']
plt.figure(figsize=(7, 5))

for i in range(3):
    # ___ tu código aquí ___
    pass

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Aleatorio')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curvas ROC One-vs-Rest — RF en Penguins')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**❓ Responde aquí:**

1. ¿Qué especie tiene el AUC más alto? ¿Coincide con la mejor F1 del reporte?
2. ¿Qué especie es más difícil de distinguir del resto? ¿Por qué?
3. Si el AUC de Chinstrap fuera 0.72, ¿qué significaría en términos prácticos?

---
## Parte 5 — Curva de Aprendizaje

In [ ]:
# 🔧 TU CÓDIGO
# Usa learning_curve con:
#   - RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
#   - train_sizes=np.linspace(0.1, 1.0, 10)
#   - cv=5, scoring='f1_weighted', n_jobs=-1
# Grafica train_mean y val_mean con bandas de std (fill_between)
# Incluye: título, xlabel, ylabel, leyenda, grid

train_sizes, train_scores, val_scores = learning_curve(
    ___________,
    X, y,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5, scoring='f1_weighted', n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

plt.figure(figsize=(8, 4))
# ___ graficar aquí ___
plt.tight_layout()
plt.show()

print(f'F1 train final:      {train_mean[-1]:.4f}')
print(f'F1 validación final: {val_mean[-1]:.4f}')
print(f'Brecha: {train_mean[-1] - val_mean[-1]:.4f}  (> 0.05 sugiere overfitting)')

**❓ Responde aquí:**

1. ¿Convergen las curvas de train y validación? ¿Qué diagnóstico hace eso?
2. ¿El modelo mejora mucho al agregar más datos o ya estabilizó?
3. Con 50 ejemplos de entrenamiento, ¿el modelo ya es confiable?

---
## 🏆 Desafío Bonus — Árbol con max_depth variable

*(Opcional — no afecta la nota)*

In [ ]:
# 🔧 DESAFÍO
# 1. Entrena DecisionTreeClassifier con max_depth en [1, 2, 3, 5, 8, None]
# 2. Para cada valor: calcula train accuracy, test accuracy y CV-5 F1-weighted
# 3. Grafica las 3 curvas en función de max_depth
# 4. Identifica el max_depth donde ocurre overfitting
# 5. ¿Cuál es el max_depth óptimo para Penguins?

depths = [1, 2, 3, 5, 8, None]
# ___ tu código aquí ___

---
## ✅ Conclusiones

*(Escribe aquí tu párrafo de conclusiones)*

Responde:
- ¿Cuál modelo elegirías para clasificar pingüinos en producción y con qué métrica lo justificas?
- ¿El modelo tiene overfitting o underfitting según la curva de aprendizaje?
- ¿Qué harías para mejorar el desempeño si las métricas no son suficientes?